In [91]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import time

prereq = "para trabajar pre req progv22025-09-15095342_procesado.xlsx"
historial = "detalle matricula sistemas pedacito 2019 cortico.xlsx"
                                

In [ ]:
class AnalizadorPrerrequisitos:
    def __init__(self):
        self.df_prereq = None
        self.df_historial = None
        self.resultado = None
        self.max_niveles_cadena = 5
        # Cache para optimización (no usado extensivamente aquí, pero lo dejo)
        self.cache_materias_estudiante = {}
        self.cache_prerrequisitos = {}
        # historial_optimizado será la tabla agregada por (Pidm, Cod materia curso)
        self.historial_optimizado = None
        # prereq_melted se crea en preparar_datos_optimizados
        self.prereq_melted = None

    def cargar_documentos(self):
        """Solicita y carga los documentos de Excel"""
        print("=== ANALIZADOR OPTIMIZADO DE PRERREQUISITOS CON CADENAS ===\n")
        
        # Solicitar archivo de prerrequisitos
        while True:
            try:
                ruta_prereq = prereq #input("Ingrese la ruta del archivo 'pre_requisitos.xlsx': ").strip()
                self.df_prereq = pd.read_excel(ruta_prereq)
                print(f"✓ Archivo de prerrequisitos cargado: {len(self.df_prereq)} registros")
                break
            except Exception as e:
                print(f"Error al cargar prerrequisitos: {e}")
                print("Intente nuevamente.\n")
        
        # Solicitar archivo de historial
        while True:
            try:
                ruta_historial = historial#input("Ingrese la ruta del archivo 'historial_asignaturas.xlsx': ").strip()
                self.df_historial = pd.read_excel(ruta_historial)
                print(f"✓ Archivo de historial cargado: {len(self.df_historial)} registros")
                break
            except Exception as e:
                print(f"Error al cargar historial: {e}")
                print("Intente nuevamente.\n")

    def validar_columnas_y_datos(self):
        """Valida que existan las columnas necesarias y verifica duplicados"""
        print("Validando estructura de datos...")

        # Columnas requeridas en prerrequisitos
        cols_prereq_req = ['Smbarul_Key_Rule', 'Programa']
        cols_prereq_faltantes = [col for col in cols_prereq_req if col not in self.df_prereq.columns]
        if cols_prereq_faltantes:
            raise ValueError(f"Faltan columnas en prerrequisitos: {cols_prereq_faltantes}")

        # Columnas requeridas en historial
        cols_hist_req = ['Codigo_Programa', 'Cod materia curso', 'Pidm', 'Materia_Aprobada', 'Calificacion_Final']
        cols_hist_faltantes = [col for col in cols_hist_req if col not in self.df_historial.columns]

        # Verificar si existe columna Periodo
        if 'Periodo' not in self.df_historial.columns:
            print("⚠️  ADVERTENCIA: No se encontró la columna 'Periodo' en el historial.")
            print("Se procederá sin validación de duplicados por periodo. Se asumirá que 'Periodo' no aplica y se contará todo el historial como antes.")
            # No rompemos: se añadirá una columna Periodo ficticia en preparar_datos_optimizados para mantener la lógica
            self.validar_duplicados_sin_periodo()
        else:
            self.validar_duplicados_con_periodo()

        if cols_hist_faltantes:
            raise ValueError(f"Faltan columnas en historial: {cols_hist_faltantes}")

        print("✓ Validación completada")

    def validar_duplicados_con_periodo(self):
        """Valida duplicados usando Pidm + Periodo + Cod materia curso"""
        print("- Validando duplicados con columna Periodo...")

        combinacion_unica = ['Pidm', 'Periodo', 'Cod materia curso']
        duplicados = self.df_historial.duplicated(subset=combinacion_unica, keep=False)

        if duplicados.any():
            registros_duplicados = self.df_historial[duplicados]
            print(f"\n⚠️  ENCONTRADOS {duplicados.sum()} REGISTROS DUPLICADOS:")
            print("Registros con la misma combinación Pidm + Periodo + Cod materia curso:")

            for _, grupo in registros_duplicados.groupby(combinacion_unica):
                print(f"- Pidm: {grupo.iloc[0]['Pidm']}, Periodo: {grupo.iloc[0]['Periodo']}, "
                      f"Materia: {grupo.iloc[0]['Cod materia curso']} ({len(grupo)} registros)")

            print(f"\nEliminando {duplicados.sum()} registros duplicados...")
            self.df_historial = self.df_historial[~duplicados].copy()
            print(f"Registros restantes: {len(self.df_historial)}")

    def validar_duplicados_sin_periodo(self):
        """Información sobre duplicados cuando no existe la columna Periodo"""
        print("- Analizando repeticiones de Pidm + Cod materia curso...")

        combinaciones = self.df_historial.groupby(['Pidm', 'Cod materia curso']).size()
        repeticiones = combinaciones[combinaciones > 1]

        if len(repeticiones) > 0:
            print(f"ℹ️  Se encontraron {len(repeticiones)} combinaciones estudiante-materia con múltiples registros.")
            print("Esto es normal ya que los estudiantes pueden tomar la misma materia varias veces.")
            print(f"Total de registros extra por repeticiones: {repeticiones.sum() - len(repeticiones)}")
        else:
            print("ℹ️  No se encontraron repeticiones en Pidm + Cod materia curso.")

    def obtener_columnas_opciones(self):
        """Obtiene las columnas de opciones de prerrequisitos (Opcion_Prereq_1 a Opcion_Prereq_21)"""
        patron = r'^Opcion_Prereq_\d+$'
        columnas_opciones = [col for col in self.df_prereq.columns if re.match(patron, col)]
        columnas_opciones.sort(key=lambda x: int(x.split('_')[-1]))
        return columnas_opciones

    def parsear_prerrequisito(self, prereq_str):
        """Parsea una cadena de prerrequisito y devuelve una lista de códigos de asignaturas"""
        if pd.isna(prereq_str) or str(prereq_str).strip() == '':
            return []

        prereq_str = str(prereq_str).strip()

        if '&' in prereq_str:
            codigos = [codigo.strip() for codigo in prereq_str.split('&')]
            return [codigo for codigo in codigos if codigo]
        else:
            return [prereq_str] if prereq_str else []

    def preparar_datos_optimizados(self):
        """Prepara estructuras optimizadas:
        - Asegura Periodo.
        - Crea Intento (1..n) por (Pidm, Cod materia curso) en orden cronológico.
        - Precalcula acumulados por periodo (aprobado/nota/último intento).
        - Convierte columnas Opcion_Prereq_* a forma larga (melt/explode).
        """
        print("Preparando estructuras de datos optimizadas...")

        if 'Periodo' not in self.df_historial.columns:
            self.df_historial['Periodo'] = 999999
            print("⚠️  'Periodo' no estaba presente. Se creó una columna 'Periodo' ficticia (999999).")

        # Normaliza códigos del historial para que coincidan con prereq
        self.df_historial['Cod materia curso'] = self.df_historial['Cod materia curso'].astype(str).str.strip()

        self.df_historial['Periodo'] = self.df_historial['Periodo'].astype(int)
        # Orden estable para acumulados
        self.df_historial = self.df_historial.sort_values(['Pidm', 'Cod materia curso', 'Periodo']).copy()

        # Intento = orden del intento por estudiante-materia
        self.df_historial['Intento'] = (
            self.df_historial
            .groupby(['Pidm','Cod materia curso'])
            .cumcount() + 1
        )

        # Flags/valores para acumulados
        aprob_flag = self.df_historial['Materia_Aprobada'].astype(str).str.upper().eq('Y')

        # Acumulados por (Pidm, Cod, Periodo): 
        #  - aprobado_hasta: alguna vez aprobado hasta ese periodo (cummax)
        #  - intento_max_hasta: máx Intento hasta ese periodo
        #  - nota_ultima_hasta: última nota observada hasta ese periodo (forward fill)
        base = self.df_historial[['Pidm','Cod materia curso','Periodo','Intento','Calificacion_Final']].copy()
        base['Aprobado_Inst'] = aprob_flag

        # Para "última nota hasta periodo" necesitamos un índice temporal estable
        base = base.sort_values(['Pidm','Cod materia curso','Periodo','Intento'])
        base['Aprobado_Hasta'] = (
            base.groupby(['Pidm','Cod materia curso'])['Aprobado_Inst'].cummax()
        )
        base['Intento_Max_Hasta'] = (
            base.groupby(['Pidm','Cod materia curso'])['Intento'].cummax()
        )
        base['Nota_Ultima_Hasta'] = (
            base.groupby(['Pidm','Cod materia curso'])['Calificacion_Final'].ffill()
        )

        

        # Compactar a una sola fila por (Pidm, Cod, Periodo) (última entrada de ese periodo)
        self.hist_cum = (
            base.sort_values(['Pidm','Cod materia curso','Periodo','Intento'])
                .groupby(['Pidm','Cod materia curso','Periodo'], as_index=False)
                .agg({
                    'Aprobado_Hasta':'last',
                    'Intento_Max_Hasta':'last',
                    'Nota_Ultima_Hasta':'last'
                })
        )

        # Agregado "global" por (Pidm, Cod) (para consultas sin periodo explícito)
        agrup = self.df_historial.groupby(['Pidm','Cod materia curso'], as_index=False)
        self.historial_optimizado = (
            self.df_historial
            .groupby(['Pidm','Cod materia curso'], as_index=False)
            .agg({
                'Materia_Aprobada': lambda s: (s.astype(str).str.upper()=='Y').any(),
                'Calificacion_Final': 'last',
                'Codigo_Programa': 'first',
                'Periodo': 'count'   # este 'count' nos da los intentos
            })
            .rename(columns={
                'Materia_Aprobada': 'Aprobada',
                'Calificacion_Final': 'Nota_Final',
                'Codigo_Programa': 'Codigo_Programa',
                'Periodo': 'Intentos'
            })
        )


        # ---- Melt de prerrequisitos ----
        print("- Procesando opciones de prerrequisitos (melt)...")
        columnas_opciones = self.obtener_columnas_opciones()
        # Quedarnos solo con columnas relevantes + opciones
        cols = ['Smbarul_Key_Rule','Programa'] + columnas_opciones
        prereq = self.df_prereq.loc[:, [c for c in cols if c in self.df_prereq.columns]].copy()

        # Convertir a largo: (Smbarul_Key_Rule, Programa, Opcion, Valor)
        melted = prereq.melt(
            id_vars=['Smbarul_Key_Rule','Programa'],
            value_vars=[c for c in columnas_opciones if c in prereq.columns],
            var_name='Opcion_Col',
            value_name='Prerrequisitos_Raw'
        ).dropna(subset=['Prerrequisitos_Raw'])

        # Número de opción (extraído del nombre de columna)
        melted['Opcion_Num'] = (
            melted['Opcion_Col'].str.extract(r'(\d+)').astype(int)
        )

        # Parsear cada string "A & B" → lista de códigos
        # (vectorizado via split/str.strip + explode + re-ensamble)
        splitted = (
            melted.assign(Prerrequisitos_Raw = melted['Prerrequisitos_Raw'].astype(str))
                .assign(_tmp = melted['Prerrequisitos_Raw'].str.split('&'))
                .explode('_tmp')
        )
        splitted['_tmp'] = splitted['_tmp'].str.strip()
        # Reagrupar en listas limpias por fila original
        parsed = (splitted
                .groupby(['Smbarul_Key_Rule','Programa','Opcion_Col','Opcion_Num'], as_index=False)
                .agg({'Prerrequisitos_Raw':'first',
                        '_tmp': lambda s: [x for x in s.tolist() if x]})
                )
        parsed = parsed.rename(columns={'_tmp':'Prerrequisitos_Codigos'})
        self.prereq_melted = parsed.sort_values(['Smbarul_Key_Rule','Opcion_Num']).reset_index(drop=True)

        print(f"✓ Estructuras optimizadas listas. Opciones registradas: {len(self.prereq_melted)}")


    def verificar_opciones_prerrequisitos_vectorizado(self, estudiantes_materias):
        """
        Para cada (Pidm, materia, Periodo[, Programa]) elige la PRIMERA opción cuyos códigos
        están aprobados HASTA ese Periodo (as-of). Si ninguna, toma la primera y marca estado "no encontrado".
        """
        if estudiantes_materias.empty:
            return pd.DataFrame()

        em = estudiantes_materias.copy()
        em = em.loc[:, ~em.columns.duplicated()].copy()

        if 'Periodo' not in em.columns:
            em['Periodo'] = 999999
        em['Periodo'] = em['Periodo'].astype(int)

        # Intentar traer Programa desde historial si no viene
        if 'Programa' not in em.columns and 'Codigo_Programa' in self.df_historial.columns:
            prog_map = (self.df_historial[['Pidm','Cod materia curso','Periodo','Codigo_Programa']]
                        .drop_duplicates()
                        .rename(columns={'Codigo_Programa':'Programa'}))
            em = em.merge(prog_map, on=['Pidm','Cod materia curso','Periodo'], how='left')

        # Merge con catálogo de prerrequisitos (por programa si hay)
        if 'Programa' in em.columns and 'Programa' in self.prereq_melted.columns:
            datos = em.merge(self.prereq_melted,
                            left_on=['Cod materia curso','Programa'],
                            right_on=['Smbarul_Key_Rule','Programa'],
                            how='left')
        else:
            datos = em.merge(self.prereq_melted,
                            left_on='Cod materia curso',
                            right_on='Smbarul_Key_Rule',
                            how='left')

        # Limpiezas defensivas
        if 'Smbarul_Key_Rule' in datos.columns:
            datos = datos.rename(columns={'Smbarul_Key_Rule':'Cod_Regla'})
        datos = datos.loc[:, ~datos.columns.duplicated()]

        # Sin opciones: materia sin prerequisitos
        if 'Opcion_Num' not in datos.columns or datos['Opcion_Num'].isna().all():
            cols = ['Pidm','Cod materia curso','Periodo','Opcion_Elegida',
                    'Prerrequisitos_Codigos','Prerrequisitos_Notas','Prerrequisitos_Intentos','Estado']
            if 'Programa' in em.columns:
                cols.insert(3, 'Programa')
            return datos.assign(Opcion_Elegida=np.nan,
                                Prerrequisitos_Codigos=[[]]*len(datos),
                                Prerrequisitos_Notas=[[]]*len(datos),
                                Prerrequisitos_Intentos=[[]]*len(datos),
                                Estado='Prerrequisito cumplido')[cols].drop_duplicates(
                                subset=['Pidm','Cod materia curso','Periodo'] + (['Programa'] if 'Programa' in em.columns else [])
                            ).reset_index(drop=True)

        # Expandir a nivel de código
        datos = datos.explode('Prerrequisitos_Codigos', ignore_index=True)
        datos = datos.rename(columns={'Prerrequisitos_Codigos':'Codigo_Prereq'})
        datos = datos[datos['Codigo_Prereq'].notna()].copy()

        # Normaliza tipo del código de prerrequisito para el as-of
        datos['Codigo_Prereq'] = datos['Codigo_Prereq'].astype(str).str.strip()

        # Traer estado as-of del prerrequisito
        asof_in = datos[['Pidm','Codigo_Prereq','Periodo']].drop_duplicates()
        asof_out = self._asof_merge_hist(asof_in, 'Codigo_Prereq')  # añade Cumple/Intentos/Nota_Final

        # === DEBUG: detectar duplicados antes de merge ===
        dups = asof_out[asof_out.duplicated(subset=['Pidm','Codigo_Prereq','Periodo'], keep=False)]
        if not dups.empty:
            print("\n⚠️ DEBUG: Se encontraron duplicados en asof_out:")
            print(dups[['Pidm','Codigo_Prereq','Periodo','Cumple','Intentos','Nota_Final']])
            print(f"Total duplicados: {len(dups)}\n")
        else:
            print("✓ DEBUG: No hay duplicados en asof_out")
        
        # Filtrar fuera filas sin Codigo_Prereq
        asof_out = asof_out[asof_out['Codigo_Prereq'].notna()].copy()
        # 🔑 Deduplicar llaves antes del merge final
        asof_out = asof_out.drop_duplicates(subset=['Pidm','Codigo_Prereq','Periodo'])

        datos = datos.merge(asof_out, on=['Pidm','Codigo_Prereq','Periodo'], how='left', validate='m:1')

        # Encontrada/Cumple
        datos['Encontrada'] = datos['Cumple'].fillna(False) | datos['Nota_Final'].notna()
        datos['Cumple'] = datos['Cumple'].fillna(False)

        # Agregar por opción
        group_keys = ['Pidm','Cod materia curso','Periodo','Opcion_Num']
        if 'Programa' in datos.columns:
            group_keys.insert(3, 'Programa')

        opciones = (datos
            .groupby(group_keys, as_index=False)
            .agg(
                Cumple=('Cumple','all'),
                Encontrada=('Encontrada','all'),
                Prerrequisitos_Codigos=('Codigo_Prereq', list),
                Prerrequisitos_Notas=('Nota_Final', list),
                Prerrequisitos_Intentos=('Intentos', list)
            )
        )

        # Selección primera que cumple; si ninguna, primera listada
        claves = ['Pidm','Cod materia curso','Periodo']
        if 'Programa' in opciones.columns:
            claves.insert(3, 'Programa')

        candidatas = (opciones[opciones['Cumple'] & opciones['Encontrada']]
                    .sort_values(claves+['Opcion_Num'])
                    .drop_duplicates(claves, keep='first'))

        primeras = (opciones.sort_values(claves+['Opcion_Num'])
                    .drop_duplicates(claves, keep='first'))

        elegido = primeras.merge(
            candidatas[claves+['Opcion_Num']].rename(columns={'Opcion_Num':'Opcion_OK'}),
            on=claves, how='left'
        )
        elegido['Opcion_Elegida'] = np.where(elegido['Opcion_OK'].notna(),
                                            elegido['Opcion_OK'], elegido['Opcion_Num']).astype(int)

        resultado = elegido.drop(columns=['Opcion_Num','Opcion_OK']).merge(
            opciones,
            left_on=claves+['Opcion_Elegida'],
            right_on=claves+['Opcion_Num'],
            how='left'
        ).drop(columns=['Opcion_Num'])

        print("Columnas en elegido:", elegido.columns.tolist())
        print("Columnas en opciones:", opciones.columns.tolist())

        for col in ['Cumple','Encontrada','Prerrequisitos_Codigos','Prerrequisitos_Notas','Prerrequisitos_Intentos']:
            if col not in resultado.columns:
                if col in ['Cumple','Encontrada']:
                    resultado[col] = False
                else:
                    resultado[col] = [[] for _ in range(len(resultado))]

        resultado['Estado'] = np.where(resultado['Cumple'] & resultado['Encontrada'],
                                    'Prerrequisito cumplido',
                                    'Tiene prerrequisito y no se encontró en la historia')

        cols_finales = claves + ['Opcion_Elegida','Prerrequisitos_Codigos',
                                'Prerrequisitos_Notas','Prerrequisitos_Intentos','Estado']
        return resultado[cols_finales].reset_index(drop=True)




    def construir_cadenas_por_nivel(self, df_base, nivel=1, visited=None):
        """
        Construye cadena por niveles (hasta self.max_niveles_cadena), sin repetir
        (Pidm, Materia_Original, Cod prereq, Periodo). Arrastra Programa si existe.
        """
        if visited is None:
            visited = set()
        if df_base is None or df_base.empty or nivel > self.max_niveles_cadena:
            return pd.DataFrame()

        df_base = df_base.loc[:, ~df_base.columns.duplicated()].copy()
        if 'Materia_Original' not in df_base.columns:
            df_base['Materia_Original'] = df_base['Cod materia curso']

        # Explode directos de este nivel
        work = df_base.copy()
        work['Posicion'] = work['Prerrequisitos_Codigos'].apply(
            lambda x: list(range(1, len(x)+1)) if isinstance(x, list) else []
        )
        work = work.explode(
            ['Prerrequisitos_Codigos','Prerrequisitos_Notas',
            'Prerrequisitos_Intentos','Posicion'],
            ignore_index=True
        )
        if work.empty:
            return pd.DataFrame()

        work = work.rename(columns={
            'Prerrequisitos_Codigos': 'Cod materia curso',
            'Prerrequisitos_Notas': 'Nota_Nivel',
            'Prerrequisitos_Intentos': 'Intentos_Nivel'
        })
        work['Nivel'] = nivel
        work['Estado_Nivel'] = np.where(work['Nota_Nivel'].notna(), 'Aprobada', 'No Aprobada')

        # Dedup de cadena por Materia_Original
        key_tuples = list(work[['Pidm','Materia_Original','Cod materia curso','Periodo']].itertuples(index=False, name=None))
        mask_nuevo = ~pd.Series(key_tuples, index=work.index).isin(visited)
        work = work.loc[mask_nuevo].copy()
        visited.update([k for k, keep in zip(key_tuples, mask_nuevo) if keep])

        if work.empty:
            return pd.DataFrame()

        # Preparar input del siguiente nivel (arrastra Programa si existe)
        keep_next = ['Pidm','Cod materia curso','Periodo']
        if 'Programa' in work.columns:
            keep_next.append('Programa')
        sig_input = work[keep_next].drop_duplicates()

        sig_eval = self.verificar_opciones_prerrequisitos_vectorizado(sig_input)

        # Mapear al "árbol": prerrequisitos de estos códigos
        sig_eval = sig_eval.rename(columns={
            'Cod materia curso': 'Cod_materia_siguiente',
            'Prerrequisitos_Codigos': 'Prerrequisitos_Codigos_sig',
            'Prerrequisitos_Notas': 'Prerrequisitos_Notas_sig',
            'Prerrequisitos_Intentos': 'Prerrequisitos_Intentos_sig'
        })

        # ---------- BLOQUE NUEVO: evitar choques de nombres en el merge del siguiente nivel ----------
        # Mapa para arrastrar la materia original del "padre"
        mapa_cols = ['Pidm','Cod materia curso','Periodo','Materia_Original']
        if 'Programa' in work.columns:
            mapa_cols.append('Programa')

        # Seleccionar SOLO las columnas existentes (por si acaso)
        mapa_cols = [c for c in mapa_cols if c in work.columns]
        mapa = work[mapa_cols].drop_duplicates().copy()

        # Quitar columnas duplicadas si vinieran de iteraciones previas
        if mapa.columns.duplicated().any():
            mapa = mapa.loc[:, ~mapa.columns.duplicated()].copy()

        # Renombrar SIEMPRE las del lado derecho para que NUNCA choquen
        rename_right = {}
        if 'Cod materia curso' in mapa.columns:
            rename_right['Cod materia curso'] = 'Cod_materia_padre'
        if 'Materia_Original' in mapa.columns:
            rename_right['Materia_Original'] = 'Materia_Original_padre'
        if rename_right:
            mapa = mapa.rename(columns=rename_right)

        # Asegurar unicidad de nombres tras el rename
        if mapa.columns.duplicated().any():
            mapa = mapa.loc[:, ~mapa.columns.duplicated()].copy()

        # Preparar claves de unión (izq = sig_eval, der = mapa)
        left_keys  = ['Pidm','Cod_materia_siguiente','Periodo']
        right_keys = ['Pidm','Cod_materia_padre','Periodo']
        if 'Programa' in sig_eval.columns and 'Programa' in mapa.columns:
            left_keys.append('Programa')
            right_keys.append('Programa')

        # MERGE limpio con sufijos controlados y deduplicación de columnas
        siguiente_nivel = sig_eval.merge(
            mapa,
            left_on=left_keys,
            right_on=right_keys,
            how='left',
            suffixes=('', '__mapa')
        )

        # Si aún así algo dejó nombres repetidos, limpiarlos
        if siguiente_nivel.columns.duplicated().any():
            siguiente_nivel = siguiente_nivel.loc[:, ~siguiente_nivel.columns.duplicated()].copy()

        # Validar que existan las columnas esperadas para continuar el árbol
        expected = ['Cod_materia_siguiente',
                    'Prerrequisitos_Codigos_sig',
                    'Prerrequisitos_Notas_sig',
                    'Prerrequisitos_Intentos_sig']
        if siguiente_nivel.empty or not set(expected).issubset(siguiente_nivel.columns):
            return work[['Pidm','Materia_Original','Cod materia curso','Nivel','Posicion',
                         'Nota_Nivel','Intentos_Nivel','Estado_Nivel','Periodo'] +
                        (['Programa'] if 'Programa' in work.columns else [])]

        # Construir df_base del próximo nivel (solo columnas necesarias y SIN nombres duplicados)
        next_cols = ['Pidm','Cod_materia_siguiente','Periodo',
                     'Materia_Original_padre',
                     'Prerrequisitos_Codigos_sig','Prerrequisitos_Notas_sig','Prerrequisitos_Intentos_sig'] + \
                    (['Programa'] if 'Programa' in siguiente_nivel.columns else [])

        # Algunas columnas podrían no existir si no hubo match; filtrar a las presentes
        next_cols = [c for c in next_cols if c in siguiente_nivel.columns]

        siguiente_df_base = (
            siguiente_nivel[next_cols]
            .rename(columns={
                'Cod_materia_siguiente'        : 'Cod materia curso',
                'Materia_Original_padre'       : 'Materia_Original',
                'Prerrequisitos_Codigos_sig'   : 'Prerrequisitos_Codigos',
                'Prerrequisitos_Notas_sig'     : 'Prerrequisitos_Notas',
                'Prerrequisitos_Intentos_sig'  : 'Prerrequisitos_Intentos'
            })
        )

        # Si por algún motivo no quedó 'Materia_Original' (sin match), heredar desde la propia materia
        if 'Materia_Original' not in siguiente_df_base.columns and 'Cod materia curso' in work.columns:
            tmp = work[['Pidm','Periodo','Cod materia curso','Materia_Original']].drop_duplicates()
            siguiente_df_base = siguiente_df_base.merge(
                tmp.rename(columns={'Cod materia curso':'__cod__'}),
                left_on=['Pidm','Periodo'],
                right_on=['Pidm','Periodo'],
                how='left'
            )
            # si no existe 'Materia_Original' aún, usar la de tmp
            if 'Materia_Original' not in siguiente_df_base.columns and 'Materia_Original' in tmp.columns:
                siguiente_df_base['Materia_Original'] = tmp['Materia_Original']
            if '__cod__' in siguiente_df_base.columns:
                siguiente_df_base = siguiente_df_base.drop(columns=['__cod__'])

        # Última protección contra columnas duplicadas antes de recursión
        if siguiente_df_base.columns.duplicated().any():
            siguiente_df_base = siguiente_df_base.loc[:, ~siguiente_df_base.columns.duplicated()].copy()
        # ---------- FIN BLOQUE NUEVO ----------


    def _asof_merge_hist(self, df, col_codigo):
        """
        Une df con self.hist_cum trayendo el último estado <= Periodo (as-of) por (Pidm, col_codigo).
        Requisitos: df tiene ['Pidm', col_codigo, 'Periodo'].
        Devuelve df con columnas añadidas: Cumple, Intentos, Nota_Final.
        Incluye fallback por grupo si merge_asof global detecta desorden.
        """
        # --- Preparar right (hist) con mismos nombres/tipos que left ---
        hist = (self.hist_cum
                .rename(columns={
                    'Cod materia curso': col_codigo,
                    'Aprobado_Hasta': 'Cumple',
                    'Intento_Max_Hasta': 'Intentos',
                    'Nota_Ultima_Hasta': 'Nota_Final'
                })[['Pidm', col_codigo, 'Periodo', 'Cumple', 'Intentos', 'Nota_Final']]
                .copy())

        # Normalización de tipos y limpieza (ambos lados)
        df = df.copy()
        # Tipos
        df['Pidm'] = pd.to_numeric(df['Pidm'], errors='coerce').astype('Int64')
        hist['Pidm'] = pd.to_numeric(hist['Pidm'], errors='coerce').astype('Int64')

        df['Periodo'] = pd.to_numeric(df['Periodo'], errors='coerce').astype('Int64')
        hist['Periodo'] = pd.to_numeric(hist['Periodo'], errors='coerce').astype('Int64')

        # col_codigo como string limpio
        df[col_codigo] = df[col_codigo].astype(str).str.strip()
        hist[col_codigo] = hist[col_codigo].astype(str).str.strip()

        # Quitar filas con NaN en claves 'by'
        df = df.dropna(subset=['Pidm', col_codigo])
        hist = hist.dropna(subset=['Pidm', col_codigo])

        # Orden idéntico en ambos
        sort_keys = ['Pidm', col_codigo, 'Periodo']
        df = df.sort_values(sort_keys).reset_index(drop=True)
        hist = hist.sort_values(sort_keys).reset_index(drop=True)

        # Si alguno está vacío, devolvemos df con NaNs en métricas
        if df.empty or hist.empty:
            df['Cumple'] = pd.NA
            df['Intentos'] = pd.NA
            df['Nota_Final'] = pd.NA
            return df

        # --- Intento global (rápido) ---
        try:
            out = pd.merge_asof(
                left=df,
                right=hist,
                by=['Pidm', col_codigo],
                left_on='Periodo',
                right_on='Periodo',
                direction='backward',
                allow_exact_matches=True
            )
            return out
        except ValueError as e:
            # Si persiste el error de orden, usar fallback por grupo
            if "keys must be sorted" not in str(e):
                raise

        # --- Fallback por grupo (más lento pero robusto) ---
        parts = []
        # No reordenes el groupby (sort=False); ordenamos dentro
        for (pidm, codigo), g_left in df.groupby(['Pidm', col_codigo], sort=False):
            g_right = hist[(hist['Pidm'] == pidm) & (hist[col_codigo] == codigo)]
            if g_right.empty:
                g = g_left.copy()
                g['Cumple'] = pd.NA
                g['Intentos'] = pd.NA
                g['Nota_Final'] = pd.NA
                parts.append(g)
                continue

            g_left_sorted = g_left.sort_values('Periodo').reset_index(drop=True)
            g_right_sorted = g_right.sort_values('Periodo').reset_index(drop=True)

            g_merged = pd.merge_asof(
                left=g_left_sorted,
                right=g_right_sorted,
                left_on='Periodo',
                right_on='Periodo',
                direction='backward',
                allow_exact_matches=True
            )
            parts.append(g_merged)

        out = pd.concat(parts, ignore_index=True)

         # 🔍 Garantizar columnas aunque vengan vacías
        for col in ['Cumple','Intentos','Nota_Final']:
            if col not in out.columns:
                out[col] = np.nan

        return out





    def _combinar_directos_y_cadena_en_resultado(self, prereq_directos):
        """
        Para cada (Pidm, materia, Periodo): une directos (Nivel=1, Tipo=Directo) + cadena (Nivel>1, Tipo=Cadena)
        SIN repetir códigos (prioriza directos). Rellena listas en self.resultado.
        """
        id_cols = ['Pidm','Cod materia curso','Periodo']

        # --- Directos en largo
        dir_long = prereq_directos.copy()
        dir_long['Prerrequisitos_Niveles'] = dir_long['Prerrequisitos_Codigos'].apply(
            lambda L: [1]*len(L) if isinstance(L, list) else []
        )
        dir_long['Prerrequisitos_Tipos'] = dir_long['Prerrequisitos_Codigos'].apply(
            lambda L: ['Directo']*len(L) if isinstance(L, list) else []
        )
        dir_long['Pos'] = dir_long['Prerrequisitos_Codigos'].apply(lambda x: list(range(1,len(x)+1)) if isinstance(x,list) else [])
        dir_long = dir_long.explode(
            ['Prerrequisitos_Codigos','Prerrequisitos_Notas','Prerrequisitos_Intentos',
            'Prerrequisitos_Niveles','Prerrequisitos_Tipos','Pos'],
            ignore_index=True
        )
        if not dir_long.empty:
            dir_long = dir_long.rename(columns={
                'Prerrequisitos_Codigos':'Codigo',
                'Prerrequisitos_Notas':'Nota',
                'Prerrequisitos_Intentos':'Intentos',
                'Prerrequisitos_Niveles':'Nivel',
                'Prerrequisitos_Tipos':'Tipo'
            })
            dir_long['EsCadena'] = dir_long['Nivel'].apply(lambda n: 'VERDADERO' if n and n>1 else 'FALSO')
            dir_long['__is_direct'] = 1
            dir_long = dir_long[id_cols + ['Codigo','Nota','Intentos','Nivel','Tipo','EsCadena','Pos','__is_direct']]
        else:
            dir_long = pd.DataFrame(columns=id_cols + ['Codigo','Nota','Intentos','Nivel','Tipo','EsCadena','Pos','__is_direct'])

        # --- Cadena (solo nivel>1)
        cad = self.cadenas if isinstance(self.cadenas, pd.DataFrame) else pd.DataFrame()
        if not cad.empty:
            cad_long = cad[cad['Nivel']>1].rename(columns={
                'Materia_Original':'_Materia_Original',
                'Cod materia curso':'Codigo',
                'Nota_Nivel':'Nota',
                'Intentos_Nivel':'Intentos',
                'Posicion':'Pos'
            })
            # Usamos _Materia_Original como "id" de materia padre
            cad_long = cad_long.rename(columns={'_Materia_Original':'Cod materia curso'})
            cad_long['Tipo'] = 'Cadena'
            cad_long['EsCadena'] = 'VERDADERO'
            cad_long = cad_long[id_cols + ['Codigo','Nota','Intentos','Nivel','Tipo','EsCadena','Pos']]
            cad_long['__is_direct'] = 0
        else:
            cad_long = pd.DataFrame(columns=id_cols + ['Codigo','Nota','Intentos','Nivel','Tipo','EsCadena','Pos','__is_direct'])

        # Alinear columnas y columnas únicas
        for df_ in (dir_long, cad_long):
            df_.columns = pd.Index(df_.columns)
        dir_long = dir_long.loc[:, ~dir_long.columns.duplicated()]
        cad_long = cad_long.loc[:, ~cad_long.columns.duplicated()]

        common_cols = list(set(dir_long.columns) | set(cad_long.columns))
        dir_long = dir_long.reindex(columns=common_cols)
        cad_long = cad_long.reindex(columns=common_cols)

        comb = pd.concat([dir_long, cad_long], ignore_index=True)

        if comb.empty:
            # inicializa listas vacías
            for col in ['Prerrequisitos_Codigos','Prerrequisitos_Notas','Prerrequisitos_Intentos','Prerrequisitos_Niveles','Prerrequisitos_Tipos','Prerrequisitos_EsCadena']:
                self.resultado[col] = [[] for _ in range(len(self.resultado))]
            return

        # Orden: directos primero, luego cadena por nivel/pos
        comb = comb.sort_values(id_cols + ['__is_direct','Nivel','Pos'])

        # Quitar códigos repetidos por id (conserva el primero → directo tiene prioridad)
        comb = comb.drop_duplicates(subset=id_cols + ['Codigo'], keep='first')

        agg = (comb.groupby(id_cols, as_index=False)
            .agg({'Codigo':list,'Nota':list,'Intentos':list,'Nivel':list,'Tipo':list,'EsCadena':list}))

        # Limpiar si existen restos
        for col in ['Prerrequisitos_Codigos','Prerrequisitos_Notas','Prerrequisitos_Intentos',
                    'Prerrequisitos_Niveles','Prerrequisitos_Tipos','Prerrequisitos_EsCadena']:
            if col in self.resultado.columns:
                self.resultado.drop(columns=[col], inplace=True)

        self.resultado = self.resultado.merge(
            agg.rename(columns={
                'Codigo':'Prerrequisitos_Codigos',
                'Nota':'Prerrequisitos_Notas',
                'Intentos':'Prerrequisitos_Intentos',
                'Nivel':'Prerrequisitos_Niveles',
                'Tipo':'Prerrequisitos_Tipos',
                'EsCadena':'Prerrequisitos_EsCadena'
            }),
            on=id_cols, how='left'
        )



    def procesar_prerrequisitos(self):
        """
        - Combina únicos (Pidm, materia, Periodo[, Programa]).
        - Evalúa directos con as-of (primera opción que cumple).
        - Construye cadena por niveles.
        - Combina directos + cadena SIN duplicar códigos (directos primero).
        - Expande a columnas UNA vez, preservando columnas.
        """
        print("Procesando prerrequisitos...")

        base_cols = ['Pidm','Cod materia curso','Periodo']
        estudiantes_materias = self.df_historial[base_cols + (['Codigo_Programa'] if 'Codigo_Programa' in self.df_historial.columns else [])]\
                                .drop_duplicates().reset_index(drop=True)
        if 'Codigo_Programa' in estudiantes_materias.columns:
            estudiantes_materias = estudiantes_materias.rename(columns={'Codigo_Programa':'Programa'})

        prereq_directos = self.verificar_opciones_prerrequisitos_vectorizado(estudiantes_materias)

        # Intentos de la materia principal hasta el periodo
        intent_principal = (self.hist_cum.rename(columns={'Intento_Max_Hasta':'Intentos_Materia_Principal'})
                            [['Pidm','Cod materia curso','Periodo','Intentos_Materia_Principal']])

        # Ensamblar self.resultado base
        self.resultado = prereq_directos.merge(intent_principal, on=['Pidm','Cod materia curso','Periodo'], how='left')

        # Cadena (largo)
        prereq_directos = prereq_directos.loc[:, ~prereq_directos.columns.duplicated()].copy()
        self.cadenas = self.construir_cadenas_por_nivel(
            prereq_directos[['Pidm','Cod materia curso','Periodo',
                            'Prerrequisitos_Codigos','Prerrequisitos_Notas','Prerrequisitos_Intentos'] + 
                            (['Programa'] if 'Programa' in prereq_directos.columns else [])]
        )

        # Unir directos + cadena sin duplicar códigos
        self._combinar_directos_y_cadena_en_resultado(prereq_directos)

        # Expansión final a columnas
        self.expandir_prerrequisitos_a_columnas()

        print("✓ Procesamiento de prerrequisitos finalizado")





    def expandir_prerrequisitos_a_columnas(self):
        """Expande listas a columnas Prereq_i_* sin perder columnas originales."""
        if 'Prerrequisitos_Codigos' not in self.resultado.columns:
            return

        base = self.resultado[['Pidm','Cod materia curso','Periodo',
                            'Prerrequisitos_Codigos','Prerrequisitos_Notas',
                            'Prerrequisitos_Intentos','Prerrequisitos_Niveles',
                            'Prerrequisitos_Tipos'] + 
                            (['Prerrequisitos_EsCadena'] if 'Prerrequisitos_EsCadena' in self.resultado.columns else [])
                            ].copy()

        base['Pos'] = base['Prerrequisitos_Codigos'].apply(lambda x: list(range(1,len(x)+1)) if isinstance(x,list) else [])
        explode_cols = ['Prerrequisitos_Codigos','Prerrequisitos_Notas','Prerrequisitos_Intentos',
                        'Prerrequisitos_Niveles','Prerrequisitos_Tipos','Pos']
        if 'Prerrequisitos_EsCadena' in base.columns:
            explode_cols.append('Prerrequisitos_EsCadena')

        base = base.explode(explode_cols, ignore_index=True)
        if base.empty:
            return

        base = base.rename(columns={
            'Prerrequisitos_Codigos':'Codigo',
            'Prerrequisitos_Notas':'Nota',
            'Prerrequisitos_Intentos':'Intentos',
            'Prerrequisitos_Niveles':'Nivel',
            'Prerrequisitos_Tipos':'Tipo',
            'Prerrequisitos_EsCadena':'EsCadena'
        })

        id_cols = ['Pidm','Cod materia curso','Periodo']
        keep_cols = id_cols + ['Pos','Codigo','Nota','Nivel','Tipo','Intentos'] + (['EsCadena'] if 'EsCadena' in base.columns else [])
        long_df = base[keep_cols]

        wide = long_df.pivot(index=id_cols, columns='Pos')
        wide.columns = [f"Prereq_{pos}_{attr}" for attr, pos in wide.columns]
        wide = wide.reset_index()

        # Merge una sola vez: preserva todas las columnas existentes
        self.resultado = self.resultado.merge(wide, on=id_cols, how='left')

        # Contadores
        def count_list(x): return len(x) if isinstance(x, list) else 0
        self.resultado['Num_Prerrequisitos_Total'] = self.resultado['Prerrequisitos_Codigos'].apply(count_list)
        self.resultado['Num_Prerrequisitos_Directos'] = self.resultado['Prerrequisitos_Niveles'].apply(
            lambda L: sum(1 for n in L if n == 1) if isinstance(L, list) else 0
        )





    def generar_reporte(self):
        """Genera un reporte resumen del análisis"""
        if self.resultado is None:
            print("No hay resultados para reportar")
            return

        print("\n=== REPORTE DE RESULTADOS ===")

        resumen = self.resultado['Estado'].value_counts()
        print("\nResumen por estado:")
        for estado, cantidad in resumen.items():
            porcentaje = (cantidad / len(self.resultado)) * 100
            print(f"- {estado}: {cantidad} ({porcentaje:.1f}%)")

        if 'Num_Prerrequisitos_Directos' in self.resultado.columns:
            print(f"\nEstadísticas de prerrequisitos directos:")
            prereq_stats = self.resultado['Num_Prerrequisitos_Directos'].value_counts().sort_index()
            for num, cantidad in prereq_stats.items():
                print(f"- {num} prerrequisitos: {cantidad} materias")

        if 'Intentos_Materia_Principal' in self.resultado.columns:
            intentos_promedio = self.resultado['Intentos_Materia_Principal'].mean()
            print(f"\nPromedio de intentos por materia: {intentos_promedio:.2f}")

        print(f"\nTotal de registros analizados: {len(self.resultado)}")

    def guardar_resultados(self):
        """Guarda los resultados en un archivo Excel"""
        if self.resultado is None:
            print("No hay resultados para guardar")
            return

        timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
        nombre_archivo = "resultados_prerrequisitos_chatggg_cadenas_"+timestamp+".xlsx"

        try:
            # Limpiar columnas completamente vacías
            self.resultado = self.resultado.dropna(axis=1, how='all')

            self.resultado.to_excel(nombre_archivo, index=False)
            print(f"✓ Resultados guardados en: {nombre_archivo}")
            print(f"Columnas en el archivo: {len(self.resultado.columns)}")
        except Exception as e:
            print(f"Error al guardar resultados: {e}")

    def ejecutar(self):
        """Ejecuta el análisis completo optimizado"""
        try:
            self.cargar_documentos()
            self.preparar_datos_optimizados()
            self.procesar_prerrequisitos()
            self.generar_reporte()
            self.guardar_resultados()
            print("\n¡Análisis optimizado completado exitosamente!")
        except Exception as e:
            print(f"\nError durante la ejecución: {e}")
            import traceback
            traceback.print_exc()

# Función principal
def main():
    analizador = AnalizadorPrerrequisitos()
    analizador.ejecutar()

if __name__ == "__main__":
    main()


=== ANALIZADOR OPTIMIZADO DE PRERREQUISITOS CON CADENAS ===

✓ Archivo de prerrequisitos cargado: 3101 registros
✓ Archivo de historial cargado: 3912 registros
Preparando estructuras de datos optimizadas...
- Procesando opciones de prerrequisitos (melt)...
✓ Estructuras optimizadas listas. Opciones registradas: 2387
Procesando prerrequisitos...

⚠️ DEBUG: Se encontraron duplicados en asof_out:
      Pidm Codigo_Prereq  Periodo Cumple Intentos Nota_Final
0     <NA>           NaN   201930   True        1        4.3
3     <NA>           NaN   202030   True        2        3.8
6     <NA>           NaN   201930   True        1        4.1
8     <NA>           NaN   201930   True        1        3.4
9     <NA>           NaN   202010   True        1        4.1
...    ...           ...      ...    ...      ...        ...
2845  <NA>           NaN   202130   True        1        3.2
2846  <NA>           NaN   202110   True        1        3.9
2847  <NA>           NaN   202110   True        1     

C:\Users\vittorinor\AppData\Local\Temp\ipykernel_16724\3145448427.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  datos['Encontrada'] = datos['Cumple'].fillna(False) | datos['Nota_Final'].notna()
C:\Users\vittorinor\AppData\Local\Temp\ipykernel_16724\3145448427.py:328: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  datos['Cumple'] = datos['Cumple'].fillna(False)



⚠️ DEBUG: Se encontraron duplicados en asof_out:
      Pidm Codigo_Prereq  Periodo Cumple Intentos Nota_Final
0     <NA>           NaN   201930   True        1        4.3
3     <NA>           NaN   202030   True        2        3.8
6     <NA>           NaN   201930   True        1        4.1
8     <NA>           NaN   201930   True        1        3.4
9     <NA>           NaN   202010   True        1        4.1
...    ...           ...      ...    ...      ...        ...
2845  <NA>           NaN   202130   True        1        3.2
2846  <NA>           NaN   202110   True        1        3.9
2847  <NA>           NaN   202110   True        1        3.9
2848  <NA>           NaN   202130   True        1        3.9
2849  <NA>           NaN   202110   True        1        4.7

[2112 rows x 6 columns]
Total duplicados: 2112

Columnas en elegido: ['Pidm', 'Cod materia curso', 'Periodo', 'Programa', 'Opcion_Num', 'Cumple', 'Encontrada', 'Prerrequisitos_Codigos', 'Prerrequisitos_Notas', 'Prerre

C:\Users\vittorinor\AppData\Local\Temp\ipykernel_16724\3145448427.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  datos['Encontrada'] = datos['Cumple'].fillna(False) | datos['Nota_Final'].notna()
C:\Users\vittorinor\AppData\Local\Temp\ipykernel_16724\3145448427.py:328: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  datos['Cumple'] = datos['Cumple'].fillna(False)


✓ Procesamiento de prerrequisitos finalizado

=== REPORTE DE RESULTADOS ===

Resumen por estado:
- Tiene prerrequisito y no se encontró en la historia: 1779 (100.0%)

Estadísticas de prerrequisitos directos:
- 0 prerrequisitos: 1779 materias

Promedio de intentos por materia: 1.20

Total de registros analizados: 1779
✓ Resultados guardados en: resultados_prerrequisitos_cadenas_2025-09-19145649.xlsx
Columnas en el archivo: 16

¡Análisis optimizado completado exitosamente!
